In [ ]:
# ============================================
# ranking_and_comparison_experiment.ipynb
#
# [실험 목적]
# 앞서 만든 랭킹형 질문 처리(예산 차이 최솟값)를 "가장 큰/작은" 같은
# 다른 표현까지 일반화하고, 3개를 초과하는 다중 사업을 동시에 비교해야
# 하는 질문(c19)이 왜 처리가 안 되는지 원인을 찾아 해결하는 게 목적
#
# [진행 방식과 알아낸 것]
#
# 1. "예산이 가장 큰/작은" 질문 처리 신규 구현 (cell 10~22)
#    - "학교 발주 사업 중 예산이 가장 큰 곳은?" 질문이 기존 로직
#      (is_closest_budget_question)으로는 감지가 안 되는 걸 확인
#    - is_extreme_budget_question()/extract_extreme_direction() 신규
#      구현, 조건 필터(학교 등) + 메타데이터에서 직접 최댓값/최솟값 계산
#    - "사업명에 'X'가 포함된" 형태의 임의 키워드 필터도 처리하도록
#      extract_name_keyword_filter() 추가
#    - "시스템"이라는 흔한 키워드로 필터링했을 때 우연히 전체 최댓값과
#      같아서 결과가 맞아 보이지만 실제로는 조건이 무시되고 있었던
#      버그를 발견 -> extract_filter_conditions가 임의 키워드는
#      감지 못 하는 근본 원인 확인 후 수정
#
# 2. c19(6개 사업 동시 비교) 원인 조사 (cell 30~41)
#    - "다음 6개 사업의 기술평가·가격평가 비율을 비교" 질문에서 답변이
#      3개 사업만 다루는 걸 발견
#    - extract_doc_hints_multi를 직접 호출해보니 실제로는 12개 문서를
#      정확히 찾아내고 있었음을 확인 -> 진짜 원인은 doc_hints[:3]이라는
#      상수 제한이 검색된 문서 중 앞 3개만 쓰고 나머지를 버리고 있었던 것
#    - extract_n_items_to_compare()로 "다음 N개 사업" 표현을 감지해
#      doc_hints를 n+6개까지 넉넉하게 확장(단순히 n개만 자르면 노이즈
#      문서에 밀려 정답이 다시 누락되는 걸 확인해 여유분을 둠)
#    - check_required_facts의 매칭 방식(단어 단위 완전일치)이 "BIS와",
#      "ERP는"처럼 조사가 붙은 정답 문구를 못 잡는 것도 함께 확인
#
# 3. 전체 회귀 검증 및 유사 패턴 스캔 (cell 42~53)
#    - core40, rag-56, set-13 전체 재검증
#    - "다음 N개" 패턴이 core40/rag-56/set-13에 c19 외에 더 있는지
#      전수 스캔(쉼표 개수 기준으로 다중 비교 후보 탐색)
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/sprint-public-procurement-rag-assistant
!pwd

/content/sprint-public-procurement-rag-assistant
/content/sprint-public-procurement-rag-assistant


In [3]:
import sys
import types
import src.data_processing.chunking as real_chunking

import pickle

from src.retrieval.indexing import HybridIndex
from src.data_processing.chunking import Chunk

import src.config as config
from pathlib import Path
import src.retrieval.indexing as indexing_module

config.CHROMA_DIR = Path('/content/drive/MyDrive/중급 프로젝트/chroma_db')

indexing_module.CHROMA_DIR = config.CHROMA_DIR
chunking_alias = types.ModuleType('src.chunking')
chunking_alias.Chunk = real_chunking.Chunk
sys.modules['src.chunking'] = chunking_alias

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

index = HybridIndex(chunks)

[HybridIndex] parent 전략 chunk 3664개는 검색 후보에서 제외(context 확장 조회 전용) - 실제 검색 대상 14575개
[embeddings] SentenceTransformer 모델 로드 시도 중... (처음 실행이면 HuggingFace에서 모델을 내려받아 몇 분 걸릴 수 있습니다)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

[embeddings] SentenceTransformer 사용: nlpai-lab/KURE-v1 (dim=1024)
[HybridIndex] 기존 임베딩 인덱스 재사용: output/chroma_db (collection=rfp_chunks__nlpai-lab_KURE-v1, backend=nlpai-lab/KURE-v1, 검색 대상 chunk 14575개 일치, 재임베딩 건너뜀)


In [4]:
child_chunks = index._searchable_chunks
print(f"검색 대상(child) chunk 수: {len(child_chunks)}")

검색 대상(child) chunk 수: 14575


In [5]:
from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

In [6]:
seen = set()
all_filenames_with_biz = []
for c in child_chunks:
    if c.doc_id not in seen:
        seen.add(c.doc_id)
        biz = c.metadata.get('발주_기관', '')
        all_filenames_with_biz.append((c.doc_id, biz))

print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [7]:
import sys as sys2
sys2.path.append('/content/drive/MyDrive/중급 프로젝트')

from generation_prompts import SYSTEM_PROMPT_V9, METADATA_DISTINCTION_INSTRUCTION, needs_metadata_distinction
from answer_generation import ask_rfp_v9, extract_doc_hints_multi, find_relevant_keywords, extract_filter_conditions, is_school_org, is_closest_budget_question, parse_closest_budget_query, is_short_period_question, extract_period_days

print("import 성공")

import 성공


In [8]:
import json, re
from src.generation.generation import check_required_facts

DATA_DIR2 = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR2 / 'dev.refined.review-candidate.jsonl', encoding='utf-8') as f:
    core40 = [json.loads(l) for l in f if l.strip()]
with open(DATA_DIR2 / 'rag-56.draft.jsonl', encoding='utf-8') as f:
    rag56 = [json.loads(l) for l in f if l.strip()]
print(f"core40: {len(core40)}개, rag56: {len(rag56)}개")

core40: 40개, rag56: 56개


In [9]:
import os, shutil
from src.evaluation.golden_set_v3 import load_golden_set_v3

src_dir = '/content/drive/MyDrive/중급 프로젝트'
dst_dir = '/content/sprint-public-procurement-rag-assistant/data/golden_set_v3'
os.makedirs(dst_dir, exist_ok=True)
for fname in ['rag-56.draft.jsonl', 'set-13.draft.jsonl', 'document-structure-visual-qa.jsonl']:
    src = os.path.join(src_dir, fname)
    dst = os.path.join(dst_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"복사 완료: {fname}")

corpus_doc_ids = {fname for fname, _ in all_filenames_with_biz}
golden_v3 = load_golden_set_v3(corpus_doc_ids=corpus_doc_ids)
print(golden_v3.shape)

복사 완료: rag-56.draft.jsonl
복사 완료: set-13.draft.jsonl
복사 완료: document-structure-visual-qa.jsonl
[load_golden_set_v3] 79건 로드(answer/visual 66건 + set 13건). 원본 패키지의 core40(40)/corpus_analytics(10) 총 50건은 우리 코퍼스와 매칭할 방법이 없어서 제외.
[load_golden_set_v3] 참고: enabled=False 69건, review.status=draft 79건 (패키지 자체가 아직 팀 승인 전이라고 명시한 항목들 - 그래도 그대로 평가에 포함시켰음, v3_enabled/v3_review_status 컬럼으로 나중에 필터링 가능)
(79, 11)


In [10]:
q_b22 = "학교 발주 사업 중에서 예산이 가장 큰 곳은?"
print("is_closest_budget_question:", is_closest_budget_question(q_b22))
print("extract_filter_conditions:", extract_filter_conditions(q_b22))

is_closest_budget_question: False
extract_filter_conditions: {'학교': True}


In [12]:
def is_extreme_budget_question(question):
    """'예산이 가장 큰/작은 곳은?' 같은 최댓값/최솟값 질문 감지"""
    return bool(re.search(r'(가장|제일)\s*(큰|작은|높은|낮은)', question)) and ('예산' in question or '금액' in question or '사업비' in question)

print(is_extreme_budget_question("학교 발주 사업 중에서 예산이 가장 큰 곳은?"))
print(is_extreme_budget_question("재난 관련 사업 중 예산이 5억 이상인 것만 알려줘"))
print(is_extreme_budget_question("사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?"))

True
False
True


In [13]:
def extract_extreme_direction(question):
    """'가장 큰/작은' 중 어느 방향인지 판별"""
    if re.search(r'(가장|제일)\s*(큰|높은)', question):
        return 'max'
    return 'min'

def find_extreme_budget(question, all_filenames_with_biz, child_chunks, filter_conditions_func, build_filter_func):
    """조건에 맞는 문서들 중 예산이 가장 크거나 작은 것을 찾음"""
    conditions = filter_conditions_func(question)
    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    meta_filter = build_filter_func(conditions) if conditions else None

    candidates = []
    for fname, biz in all_filenames_with_biz:
        meta = doc_to_meta.get(fname, {})
        if meta_filter and not meta_filter(meta, fname):
            continue
        amt = meta.get('사업_금액')
        if amt is not None:
            candidates.append((fname, amt))

    if not candidates:
        return None

    direction = extract_extreme_direction(question)
    if direction == 'max':
        return max(candidates, key=lambda x: x[1])
    else:
        return min(candidates, key=lambda x: x[1])

# 테스트 - build_meta_filter는 ask_rfp_v9 내부에 있어서 직접 못 꺼내니, 임시로 간단 버전 재정의
def temp_build_filter(conds):
    def _filter(meta, fname=''):
        if conds.get('학교'):
            if not is_school_org(meta.get('발주_기관')):
                return False
        return True
    return _filter

result = find_extreme_budget(q_b22, all_filenames_with_biz, child_chunks, extract_filter_conditions, temp_build_filter)
print(result)

('고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 11270000000.0)


In [14]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

new_functions = '''

def is_extreme_budget_question(question):
    """'예산이 가장 큰/작은 곳은?' 같은 최댓값/최솟값 질문 감지"""
    return bool(re.search(r'(가장|제일)\\s*(큰|작은|높은|낮은)', question)) and ('예산' in question or '금액' in question or '사업비' in question)


def extract_extreme_direction(question):
    """'가장 큰/작은' 중 어느 방향인지 판별"""
    if re.search(r'(가장|제일)\\s*(큰|높은)', question):
        return 'max'
    return 'min'

'''

marker = "def ask_rfp_v9("
idx = content.find(marker)
content = content[:idx] + new_functions.strip() + "\n\n\n" + content[idx:]

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("함수 추가 완료")

함수 추가 완료


In [15]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    if is_aggregation_question(question) and len(doc_hints) >= 1:"""

new_code = """    # 조건(학교/지자체 등)에 맞는 문서 중 예산 최댓값/최솟값을 찾는 질문 우선 처리
    if is_extreme_budget_question(question):
        meta_filter_extreme = build_meta_filter(conditions) if conditions else None
        candidates_extreme = []
        for fname, biz in all_filenames_with_biz:
            meta = doc_to_meta.get(fname, {})
            if meta_filter_extreme and not meta_filter_extreme(meta, fname):
                continue
            amt = meta.get('사업_금액')
            if amt is not None:
                candidates_extreme.append((fname, amt))
        if candidates_extreme:
            direction = extract_extreme_direction(question)
            result = max(candidates_extreme, key=lambda x: x[1]) if direction == 'max' else min(candidates_extreme, key=lambda x: x[1])
            fname, amt = result
            return f"{fname} — {amt:,.0f}원\\n\\n[근거: {fname}]"

    if is_aggregation_question(question) and len(doc_hints) >= 1:"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [16]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_b22 = "학교 발주 사업 중에서 예산이 가장 큰 곳은?"
answer = ask_rfp_v9(q_b22, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf — 11,270,000,000원

[근거: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf]


In [17]:
test_extreme = [
    "사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?",
    "재난 관련 사업 중 예산이 가장 작은 곳은?",
    "전체 사업 중 예산이 가장 큰 곳은?",
]

for q in test_extreme:
    print(f"{q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp — 14,107,009,000원

[근거: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp]

재난 관련 사업 중 예산이 가장 작은 곳은?
재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp — 100,000,000원

[근거: 재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp]

전체 사업 중 예산이 가장 큰 곳은?
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp — 14,107,009,000원

[근거: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp]



In [18]:
# "시스템"이 포함된 문서들 중 진짜 최댓값이 뭔지 직접 확인
system_docs = [(fname, biz) for fname, biz in all_filenames_with_biz if '시스템' in fname]
doc_to_meta_test = {}
for c in child_chunks:
    if c.doc_id not in doc_to_meta_test:
        doc_to_meta_test[c.doc_id] = c.metadata

system_with_budget = [(fname, doc_to_meta_test.get(fname, {}).get('사업_금액')) for fname, biz in system_docs]
system_with_budget = [(f, a) for f, a in system_with_budget if a is not None]
system_with_budget.sort(key=lambda x: -x[1])
print(f"'시스템' 포함 문서 수: {len(system_with_budget)}개")
print("최댓값 top 3:", system_with_budget[:3])

'시스템' 포함 문서 수: 79개
최댓값 top 3: [('한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp', 14107009000.0), ('고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 11270000000.0), ('한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp', 2392940000.0)]


In [19]:
q = "사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?"
print("extract_filter_conditions:", extract_filter_conditions(q))

extract_filter_conditions: {}


In [20]:
def extract_name_keyword_filter(question):
    """'사업명에 'X'가 포함된' 같은 표현에서 X를 추출"""
    m = re.search(r"['\"]([^'\"]+)['\"]", question)
    if m and ('사업명' in question or '이름' in question):
        return m.group(1)
    return None

print(extract_name_keyword_filter("사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?"))
print(extract_name_keyword_filter("재난 관련 사업 중 예산이 가장 작은 곳은?"))

시스템
None


In [21]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """def extract_extreme_direction(question):
    \"\"\"'가장 큰/작은' 중 어느 방향인지 판별\"\"\"
    if re.search(r'(가장|제일)\\s*(큰|높은)', question):
        return 'max'
    return 'min'"""

new_code = """def extract_extreme_direction(question):
    \"\"\"'가장 큰/작은' 중 어느 방향인지 판별\"\"\"
    if re.search(r'(가장|제일)\\s*(큰|높은)', question):
        return 'max'
    return 'min'


def extract_name_keyword_filter(question):
    \"\"\"'사업명에 'X'가 포함된' 같은 표현에서 파일명 키워드 X를 추출\"\"\"
    m = re.search(r"['\\\"]([^'\\\"]+)['\\\"]", question)
    if m and ('사업명' in question or '이름' in question):
        return m.group(1)
    return None"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [22]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    # 조건(학교/지자체 등)에 맞는 문서 중 예산 최댓값/최솟값을 찾는 질문 우선 처리
    if is_extreme_budget_question(question):
        meta_filter_extreme = build_meta_filter(conditions) if conditions else None
        candidates_extreme = []
        for fname, biz in all_filenames_with_biz:
            meta = doc_to_meta.get(fname, {})
            if meta_filter_extreme and not meta_filter_extreme(meta, fname):
                continue
            amt = meta.get('사업_금액')
            if amt is not None:
                candidates_extreme.append((fname, amt))
        if candidates_extreme:
            direction = extract_extreme_direction(question)
            result = max(candidates_extreme, key=lambda x: x[1]) if direction == 'max' else min(candidates_extreme, key=lambda x: x[1])
            fname, amt = result
            return f"{fname} — {amt:,.0f}원\\n\\n[근거: {fname}]"
"""

new_code = """    # 조건(학교/지자체 등)에 맞는 문서 중 예산 최댓값/최솟값을 찾는 질문 우선 처리
    if is_extreme_budget_question(question):
        meta_filter_extreme = build_meta_filter(conditions) if conditions else None
        name_kw_filter = extract_name_keyword_filter(question)
        candidates_extreme = []
        for fname, biz in all_filenames_with_biz:
            meta = doc_to_meta.get(fname, {})
            if meta_filter_extreme and not meta_filter_extreme(meta, fname):
                continue
            if name_kw_filter and name_kw_filter not in fname:
                continue
            amt = meta.get('사업_금액')
            if amt is not None:
                candidates_extreme.append((fname, amt))
        if candidates_extreme:
            direction = extract_extreme_direction(question)
            result = max(candidates_extreme, key=lambda x: x[1]) if direction == 'max' else min(candidates_extreme, key=lambda x: x[1])
            fname, amt = result
            return f"{fname} — {amt:,.0f}원\\n\\n[근거: {fname}]"
"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [23]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_name_keyword_filter

importlib.reload(answer_generation)

# 위험 검증: "시스템" 대신 일부러 최댓값이 다른 키워드로 테스트
q1 = "사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?"
q2 = "사업명에 '위원회'가 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?"

for q in [q1, q2]:
    print(f"{q}")
    print("추출된 이름 필터:", extract_name_keyword_filter(q))
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

사업명에 '시스템'이 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?
추출된 이름 필터: 시스템
한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp — 14,107,009,000원

[근거: 한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp]

사업명에 '위원회'가 포함된 사업 중 예산이 가장 큰 사업은 무엇인가요?
추출된 이름 필터: 위원회
사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp — 5,031,000,000원

[근거: 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp]



In [24]:
final_ext_check_40 = []
for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_ext_check_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 243,000,000원 — 부가가치세(VAT) 포함되어 있습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)
예산: 70,000,000원 (VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
사업은 2차로 나뉩니다(1차: 계약일부터 4개월, 2차: 1차 완료일부터 2개월). 기술평가 비중 90%, 가격평가 비중 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일(문서 내 추진일정에서 계약일로부터 3개월로 표기됨, 종료예정일 예시 '2024년 11월 1일까지')  
시범 도입 규모: 1단계 3개 기관(서울 2개소, 울산 1개소)

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서와 제안서를 모두 전자적으로 나라장터(e-발주시스템, 전자입찰)를 통해 제출해야 합니다. 입찰서는 나라장터에 의해 전자적으로만 제출되어야 하며, 제안서도 나라장터(e-발주시스템)를 통해 전자 제출합니다. 제출기한은 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다
[근거: 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp, 한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp, 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp]

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다
[근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)]

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
이 요청은 제가 판단할 수 없습니다.

다만 제공된 문서에 근거한, 귀사가 스스로 대조해서 확인해야 할 참가자격 항목(체크리스트)은 아래와 같습니다.

- 지방자치단체를 당사자로 하는 계약에 관한 법령 시행령 제92조에 해당되는 부정당업자에 해당하지 않을 것.  
- 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시일 것(지방자치단체를 당사자로 하는 계약에 관한 법령 시행령 제20조 제1항 6호, 시행규칙 제24조 제2항).  
- 나라장터(G2B)에서 입찰서 제출마감일 전일까지 소프트웨어사업자(업종코드 1468, 컴퓨터관련서비스사업)로 입찰참가자격 등록을 마칠 것.  
- 대기업·중견기업 소프트웨어 사업자 및 상호출자제한기업집단 소속회사는 참여 불가(소프트웨어산업 진흥법 제48조 및 관련 지침).  
- 정보시스템개발서비스(세부품명번호 8111159901)의 직접생산확인증명서 보유(입찰마감 전일까지 발급·유효기간 내).  
- 공동수급(공동이행방식) 불허 — 단독으로 참여 가능해야 함.  
- 하도급 허용되지 않음(문서상 명시).  

위 항목 중 문서에 명확히 근거된 조건만 나열했습니다. 각 항목에 대해 귀사의 내부 서류·등록상태·사업자 유형 등을 대조해주십시오. 상세 서류 확인이 필요하거나 문서 전체 원문 검토를 원하시면 알려주세요.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-unknown-005][unknown] 서민금융 채팅상담 사업에 경쟁사가 몇 곳 참여할 예정이야?
확인되지 않습니다.

[근거: 서민

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
확인되지 않습니다.

이 문서들에서는 평가 세부항목·결과는 공개하지 않거나 비공개 원칙을 명시하고 있어, 문서 밖의 비공개 평가포인트를 제가 제공할 수 없습니다.

[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp, 부산관광공사_경영정보시스템 기능개선.hwp]

[dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
이 요청은 제공된 문서만으로 수주확률(퍼센트)을 산정할 수 없어 답변할 수 없습니다.

필요한 경우 참고할 수 있는 최소 조건 목록(간단히):
- 우리 회사의 자격요건 충족 여부(공고의 참가자격 항목 대비)
- 제출 예정 제안서 내용과 평가기준(배점) 대비 점수 추정
- 경쟁사 수 및 경쟁사의 제안 역량 정보
- 입찰 방식(제한경쟁/긴급/지역제한 등)과 평가절차 세부사항
- 과거 유사 사업 수주 실적 및 가격 제안(금액) 정보

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다.  
[근거: 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp]

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
실시간 진행률은 제공된 문서에서 확인할 수 없습니다. 계약 지연 가능성(확률)은 제공된 문서 범위에서는 확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [25]:
def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]

    if numbers:
        all_numbers_match = True
        for num in numbers:
            if num in answer_norm:
                continue
            if len(num) == 4 and num.startswith('20'):
                if num[2:] in answer_norm:
                    continue
            num_no_zero = re.sub(r'^0+', '', num)
            if num_no_zero and num_no_zero in answer_norm:
                continue
            all_numbers_match = False
            break

        if not all_numbers_match:
            return False

        raw_words = re.split(r'[\s,·:()]+', fact_text)
        stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
        core_words = []
        for w in raw_words:
            w = w.rstrip('.,')
            if len(w) < 2:
                continue
            if re.match(r'^\d', w):
                continue
            for suf in stopwords_suffix:
                if w.endswith(suf) and len(w) > len(suf):
                    w = w[:-len(suf)]
                    break
            w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
            if len(w) >= 2:
                core_words.append(w)

        if not core_words:
            return True

        match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
        return match_count / max(len(core_words), 1) >= 0.2

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는',
                    '해드릴 수 없', '드릴 수 없']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [26]:
for r in final_ext_check_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in final_ext_check_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 100.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 100.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 100.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[d

In [27]:
final_ext_check_56 = []
for item in rag56:
    question = item['question']
    answer = ask_rfp_v9(question, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_ext_check_56.append({'case_id': item['case_id'], 'answer': answer})
    print(f"[{item['case_id']}] {question}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c01] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c02] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024. 10. 31.까지 완료해야 합니다.  
[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[supplemental-qa-c03] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
사업예산: 1,515,000천원 (부가세 포함)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c04] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰  
낙찰 절차(사업자 선정 방식): 협상에 의한 계약(협상절차에 따른 선정)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c05] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월 동안 수행합니다.  
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c06] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
181,913,000원 (VAT 포함)  
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c07] 인천공항운영서비스가 회계·인사 등 경영업무를 통합할 차세대 시스템을 구축하는 데 걸리는 기간은 얼마인가요?
계약 체결일로부터 9개월(안정화 기간 포함).

[근거: 인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp]

[suppl

In [28]:
for r in final_ext_check_56:
    item = next(it for it in rag56 if it['case_id'] == r['case_id'])
    matched, total = check_required_facts(r['answer'], item['gold'].get('required_fact_groups'))
    r['score'] = round(matched / total * 100, 2) if total else None
    print(f"[{r['case_id']}] 점수: {r['score']}")

valid_scores_56 = [r['score'] for r in final_ext_check_56 if r['score'] is not None]
print(f"\nrag-56 전체 평균: {sum(valid_scores_56)/len(valid_scores_56):.2f}/100 ({len(valid_scores_56)}개)")

[supplemental-qa-c01] 점수: 100.0
[supplemental-qa-c02] 점수: 100.0
[supplemental-qa-c03] 점수: 100.0
[supplemental-qa-c04] 점수: 100.0
[supplemental-qa-c05] 점수: 100.0
[supplemental-qa-c06] 점수: 100.0
[supplemental-qa-c07] 점수: 100.0
[supplemental-qa-c08] 점수: 100.0
[supplemental-qa-c09] 점수: 100.0
[supplemental-qa-c10] 점수: 100.0
[supplemental-qa-c11] 점수: 100.0
[supplemental-qa-c12] 점수: 66.67
[supplemental-qa-c13] 점수: 100.0
[supplemental-qa-c14] 점수: 100.0
[supplemental-qa-c15] 점수: 100.0
[supplemental-qa-c16] 점수: 0.0
[supplemental-qa-c18] 점수: 100.0
[supplemental-qa-c19] 점수: 33.33
[supplemental-qa-c20] 점수: 50.0
[supplemental-qa-c23] 점수: 100.0
[supplemental-qa-c25] 점수: 50.0
[supplemental-qa-g01] 점수: 100.0
[supplemental-qa-g02] 점수: 100.0
[supplemental-qa-g03] 점수: 50.0
[supplemental-qa-g04] 점수: 100.0
[supplemental-qa-g05] 점수: 100.0
[supplemental-qa-g06] 점수: 100.0
[supplemental-qa-g07] 점수: 100.0
[supplemental-qa-g08] 점수: 80.0
[supplemental-qa-g11] 점수: 33.33
[supplemental-qa-g12] 점수: 75.0
[supplemental-q

In [30]:
final_ext_check_set13 = []
set_items_full = golden_v3[golden_v3['source_lane'] == 'set']

for _, row in set_items_full.iterrows():
    q = row['query']
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_ext_check_set13.append({'id': row['id'], 'query': q, 'answer': answer, 'expected': row['expected_doc_id']})
    print(f"[{row['id']}] {q}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b1] 철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요?
사업명: 철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역  
발주기관: 국가철도공단

[근거: 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp)]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b10] 코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요?
예약발매시스템 개량을 위한 업무 종합계획(ISMP) 수립용역, 즉 "예약발매시스템 개량 ISMP 용역"입니다.

[근거: 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp]

[supplemental-set-b12] 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘
아래 문서들에 근거한, 학교(대학교/대학/과학기술원)가 발주한 시스템 사업 목록입니다. 각 항목은 사업명 · 발주기관 · 사업예산(문서 표기) · 사업기간(문서 표기)이 있으면 함께 표기했습니다.

- 한영대학교 — 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 · 130,000,000원 (VAT 포함) · 계약일로부터 3개월 (안정화기간 1개월 포함)  
- 고려대학교 — 차세대 포털·학사 정보시스템 구축 사업 · 11,270,000,000원 (V.A.T 포함) · 계약일로부터 24개월 이내  
- 서울시립대학교 — 학업성취도 다차원 종단분석 통합시스템 1차 고도화 · 242,900,000원 (부가세 포함) · 계약체결일로부터 5개월  
- 경희대학교 — 산학협력단 정보시스템(인포21) 운영 용역업체 선정 · 400,000,000원 (부가세 포함) · 계약체결일~2026.04.30 (2년 사업, 1차년도 평가에 따라 2차년도 결정)  
- 전북대학교 — JST 공유대학(원) xAPI기반 LRS시스템 구축 · 116,000,000원 (부가세 포함) · 계약체결일로부터 75일  
- 광주과학기술원 — 학사시스템 기능개선 사업 · 157,300,000원 (VAT 포함) · 계약체결일로부터 5개월 이내  
- 광주과학기술원 — 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업 · 54,450,000원 (VAT 포함) · 계약체결일로부터 3개월 이내  
- 을지대학교 — 을지대학교 비교과시스템 개발사업 · (사업예

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b3] 병원 전산 장애나 재해에 대비해 진료정보를 복구할 체계를 마련하는 적십자 관련 용역은 무엇인가요?
적십자병원 병원정보 재해복구시스템 구축 용역입니다.  
[근거: 대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b4] 학생들의 교과 밖 활동을 관리할 온라인 환경을 새로 만드는 을지대 사업은 무엇인가요?
을지대학교 비교과시스템 개발 사업입니다.

[근거: 을지대학교_을지대학교 비교과시스템 개발.hwp]



In [31]:
def evaluate_set_answer(expected_docs, answer_text):
    all_docs_in_answer = [doc for doc, _ in all_filenames_with_biz if doc in answer_text]
    expected_set = set(expected_docs)
    found_set = set(all_docs_in_answer)
    tp = len(expected_set & found_set)
    fp = len(found_set - expected_set)
    fn = len(expected_set - found_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1, tp, fp, fn

results_set13_v2 = []
for item in final_ext_check_set13:
    p, r, f1, tp, fp, fn = evaluate_set_answer(item['expected'], item['answer'])
    results_set13_v2.append({'id': item['id'], 'precision': p, 'recall': r, 'f1': f1})
    print(f"[{item['id']}] P={p:.2f} R={r:.2f} F1={f1:.2f} (정답 {tp}개, 오탐 {fp}개, 누락 {fn}개)")

avg_p = sum(r['precision'] for r in results_set13_v2) / len(results_set13_v2)
avg_r = sum(r['recall'] for r in results_set13_v2) / len(results_set13_v2)
avg_f1 = sum(r['f1'] for r in results_set13_v2) / len(results_set13_v2)
print(f"\n평균 Precision: {avg_p:.3f}, 평균 Recall: {avg_r:.3f}, 평균 F1: {avg_f1:.3f}")

[supplemental-set-b1] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)
[supplemental-set-b10] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)
[supplemental-set-b12] P=1.00 R=1.00 F1=1.00 (정답 12개, 오탐 0개, 누락 0개)
[supplemental-set-b14] P=1.00 R=1.00 F1=1.00 (정답 5개, 오탐 0개, 누락 0개)
[supplemental-set-b15] P=0.80 R=1.00 F1=0.89 (정답 12개, 오탐 3개, 누락 0개)
[supplemental-set-b16] P=1.00 R=1.00 F1=1.00 (정답 2개, 오탐 0개, 누락 0개)
[supplemental-set-b20] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)
[supplemental-set-b21] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)
[supplemental-set-b22] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)
[supplemental-set-b23] P=1.00 R=1.00 F1=1.00 (정답 2개, 오탐 0개, 누락 0개)
[supplemental-set-b24] P=1.00 R=1.00 F1=1.00 (정답 8개, 오탐 0개, 누락 0개)
[supplemental-set-b3] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)
[supplemental-set-b4] P=1.00 R=1.00 F1=1.00 (정답 1개, 오탐 0개, 누락 0개)

평균 Precision: 0.985, 평균 Recall: 1.000, 평균 F1: 0.991


In [32]:
item_c19 = next(it for it in rag56 if it['case_id'] == 'supplemental-qa-c19')
print("질문:", item_c19['question'])
print()
print("정답:", item_c19['gold'].get('reference_answer'))
print()
print("required_fact_groups:", item_c19['gold'].get('required_fact_groups'))

질문: 다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다.

정답: 평택시 BIS와 인천공항 ERP는 기술 80%, 가격 20%입니다. 한국철도공사, 국방과학연구소, GKL, 한국농어촌공사 사업은 기술 90%, 가격 10%입니다. 각 문서에는 이 비율 차이가 발생한 이유가 명시되어 있지 않습니다.

required_fact_groups: [['평택시 BIS와 인천공항 ERP는 기술 80% 가격 20%'], ['한국철도공사 국방과학연구소 GKL 한국농어촌공사는 기술 90% 가격 10%'], ['비율 차이의 이유가 명시되어 있지 않음']]


In [35]:
docs_to_check = [
    '경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp',
    '인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp',
    '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp',
]

for doc_id in docs_to_check:
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    found = False
    for c in doc_c:
        if '80%' in c.text or '80 %' in c.text or ('기술' in c.text and '20%' in c.text):
            print(f"{doc_id[:40]}...")
            idx = c.text.find('80')
            print(c.text[max(0,idx-100):idx+150])
            print()
            found = True
            break
    if not found:
        print(f"{doc_id[:40]}... 80% 관련 텍스트 못 찾음\n")

경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp...
거 대기업 및 중견기업 소프트웨어 사업자는 본 입찰에 참여할 수 없음
다. 사업자 선정방법 및 협상절차
1)『지방자치단체 입찰시 낙찰자 결정기준』에 의거 종합배점기준을 기술제안서 80%, 가격제안서 20%로 배분ㆍ평가하여 고득점자부터 협상을 실시한다.

인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .h...
24.)을 준용
  ❍ 입찰 및 계약방식

[표]
구분 | 세 부 내 용 | 비고
입찰방식 | 제한경쟁입찰
낙찰자 결정 | 협상에 의한 계약
평가방식 | 제안서 평가 | 기술평가(80%) + 가격평가(20%)

그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.h...
에는 적용되지 않으며, 한 개 이상의 큰 이미지(이미지 500KB이상) 및 동영상을 가지고 있는 페이지에는 적용되지 않음ㅇ또한 시스템을 사용하는 사용자 숫자가 동시 사용자 용량의 80%를초과하는 경우에는 적용되지 않음ㅇ 적정 디스플레이 시간, 기준 데이터는 기관과 협의하여 조정 가능□최종 사용자 응답시간은 네트워크 시간까지 포함해야 하며, 기관내에 위치한 사용자의 네트워크 대역폭은 유선랜 100M bps를 기준으로 함□서비스 속도가 중요한 주요



In [36]:
doc_c = [c for c in child_chunks if c.doc_id == '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp']
for c in doc_c:
    if '기술평가' in c.text and ('가격평가' in c.text or '%' in c.text):
        idx = c.text.find('기술평가')
        print(c.text[max(0,idx-50):idx+150])
        print()

00천원 (부가세 포함)
□ 계약방법 : 제한경쟁입찰(협상에 의한 계약)
□ 평가방식 : 기술평가(90%)/가격평가(10%)

2. 추진 배경 및 필요성

□ (시스템 노후화) 그룹웨어 및 기록물관리, 사내SNS(별별얘기), 메신저 시스템 노후화로 SW의 기술지원 종료, 요구사항 증가로 인한 설계 한계 초과 및 서비스 질 저하 등 개선 필요

□ (정부권장

직중인 자 이어야 함

5. 제안서 제출 및 평가 방법

 가. 제안서 평가기준

  □ 기술평가와 가격평가를 실시하여 종합평가점수로 평가
  □ 평가배점 : 기술평가(90점), 가격평가(10점)

  *「행정기관 및 공공기관 정보시스템 구축·운영 지침」제18조제1항에 따라 기술평가 배점한도를 90점으로 함

 ㅇ 기술평가 : 정량평가(10점), 정성평가(

」(과학기술정보통신부고시) 준용

 나. 우선협상대상자 선정방법

  □ 제안서 평가결과 기술평가(90점) 점수가 85%(76.5점) 이상인 자를 대상으로 가격평가(10점)점수를 합산 하여 고득점 순으로 협상순위 결정

 ㅇ 합산점수가 동일한 경우 우선협상 대상자 지정은 기술평가 점수의 순서에 따라 정하고, 기술평가 점수가 동일한 경우에는 기술평가의 배점이



In [37]:
q_c19 = "다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다."

hints_c19 = extract_doc_hints_multi(q_c19, all_filenames_with_biz)
print(f"현재 로직으로 찾은 문서 힌트 ({len(hints_c19)}개):")
for h in hints_c19:
    print(f"  {h}")

현재 로직으로 찾은 문서 힌트 (12개):
  국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp
  한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
  한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp
  인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp
  그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp
  국방과학연구소_대용량 자료전송시스템 고도화.hwp
  한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp
  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
  한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp
  경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp
  사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp
  한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp


In [38]:
print("잘린 후(실제 사용되는) 힌트:")
print(hints_c19[:3])

잘린 후(실제 사용되는) 힌트:
['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp', '한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp']


In [39]:
def extract_n_items_to_compare(question):
    """'다음 N개 사업을' 같은 표현에서 N을 추출"""
    m = re.search(r'다음\s*(\d+)\s*개\s*(사업|문서|기관)', question)
    if m:
        return int(m.group(1))
    return None

print(extract_n_items_to_compare(q_c19))

6


In [40]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

new_functions = '''

def extract_n_items_to_compare(question):
    """'다음 N개 사업을' 같은 표현에서 N을 추출"""
    m = re.search(r'다음\\s*(\\d+)\\s*개\\s*(사업|문서|기관)', question)
    if m:
        return int(m.group(1))
    return None

'''

marker = "def ask_rfp_v9("
idx = content.find(marker)
content = content[:idx] + new_functions.strip() + "\n\n\n" + content[idx:]

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("함수 추가 완료")

함수 추가 완료


In [41]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]"""

new_code = """    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    n_items_to_compare = extract_n_items_to_compare(question)
    if n_items_to_compare is not None:
        doc_hints = doc_hints[:max(n_items_to_compare, 3)]
    else:
        doc_hints = doc_hints[:3]"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [42]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_c19 = "다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다."

answer = ask_rfp_v9(q_c19, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

요약(그룹화)
- 기술평가 90% / 가격평가 10%: 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역
- 기술평가 80% / 가격평가 20%: 인천공항운영서비스 차세대 ERP 시스템 구축사업
- 확인 불가: 평택시 BIS 구축사업 — 관련 문서가 제공되지 않아 평가비율 확인 불가

각 사업별 평가비율 및 문서 내 근거(문서에 '비율의 이유'가 명시되어 있는지)
1) 한국철도공사 운행정보기록 자동분석시스템 개량 사업  
   - 비율: 기술평가 90%, 가격평가 10%  
   - 문서에 비율 근거 표기 여부: 예(문서에 평가비율과 기술평가 방법/기준 설명 포함)  

2) 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업  
   - 비율: 기술평가 90%, 가격평가 10%  
   - 문서에 비율 근거 표기 여부: 예(평가비율 명시 및 소프트웨어 기술성 평가기준·협상기준 준용 언급)  

3) GKL(그랜드코리아레저) 그룹웨어 구축사업  
   - 비율: 기술평가 90%, 가격평가 10%  
   - 문서에 비율 근거 표기 여부: 예(평가비율 명시 및 준용 지침 언급)  

4) 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역  
   - 비율: 기술평가 90%, 가격평가 10%  
   - 문서에 비율 근거 표기 여부: 예(평가비율 명시 및 협상에 의한 계약·기술성 평가기준 준용 언급)  

5) 인천공항운영서비스 차세대 ERP 시스템 구축사업  
   - 비율: 기술평가 80%, 가격평가 20%  
   - 문서에 비율 근거 표기 여부: 예(문서에 평가비율 명시 및 적용하는 법령/지침 표기)  

6) 평택시 BIS 구축사업  
   - 비율: 확인되지 않습니다 (관련 문서가 제공되지 않음)

비율 차이에 대한 설명(문서에 근거가 있는지)
- 제공된 문서들은 모두 각 기관이 준용

In [43]:
print(hints_c19[:6])

['국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp', '한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp', '인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp', '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp', '국방과학연구소_대용량 자료전송시스템 고도화.hwp']


In [44]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    n_items_to_compare = extract_n_items_to_compare(question)
    if n_items_to_compare is not None:
        doc_hints = doc_hints[:max(n_items_to_compare, 3)]
    else:
        doc_hints = doc_hints[:3]"""

new_code = """    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    n_items_to_compare = extract_n_items_to_compare(question)
    if n_items_to_compare is not None:
        doc_hints = doc_hints[:n_items_to_compare + 6]
    else:
        doc_hints = doc_hints[:3]"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [45]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_c19 = "다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다."

answer = ask_rfp_v9(q_c19, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

요약(동일 평가비율끼리 묶음) — 기술평가 : 가격평가
- 90% : 10% (5건)
  - 국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp — 90:10
  - 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp — 90:10
  - 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp — 90:10
  - 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp — 90:10
  - (문서들에서 동일한 90:10 배율 명시)

- 80% : 20% (2건)
  - 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp — 80:20
  - 인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp — 80:20

문서에 나타난 “비율 차이의 이유” 여부
- 국방과학연구소_기록관리시스템… : 이유(명시적 근거)는 문서에서 직접적 설명 대신 평가방식·기준(예: 기술성 평가·기술위원회 구성, 소프트웨어 기술성 평가기준 준용 등)을 근거로 제시. → 명확한 '비율 선정 이유' 문장 없음.
- 한국철도공사_운행정보기록 자동분석시스 : 평가비율은 문서에 명시되어 있음(90:10). 다만 비율을 선택한 구체적 사유는 별도로 설명되어 있지 않고 관련 규정(협상에 의한 계약 기준 등)을 준용한다고 표기. → 명확한 이유 설명 없음.
- 그랜드코리아레저(GKL)_그룹웨어 : 기술평가 90점 배점(기술중시) 및 관련 지침(행정기관‧공공기관 지침, 소프트웨어 기술성 평가기준 등)을 준용한다고 명시. '기술중시'로 90점을 적용한 근거로 지침 준용을 언급하나 별도의 상세 논리(사업특성에 따른 선택 이유)는 없음. → 간접적 근거(지침 준용) 제시.
- 한국농어촌공사_네팔 Pilot : 기술평가 90:10으로 명시, 평가방법에서 공사·지침 준용을 언급. 구체적 비율 선택 사유는 문서에 상세히 서술되어 있지 않음. → 간접적 근거(지침 준용) 제시.
- 경기도 평

In [46]:
final_c19_check_40 = []
for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_c19_check_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 금 243,000,000원 — 부가가치세(VAT) 포함되어 있음.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)
예산: 70,000,000원(금칠천만원, VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차(1차: 시스템 구축 및 초기 데이터 구축, 2차: 리포팅툴 및 출력양식 개발)로 나뉩니다.
- 평가 비중은 기술평가 90%, 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일까지.

시범 도입 규모: 1단계 3개 기관(서울 2개소, 울산 1개소).

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서와 제안서는 전자적으로 나라장터(e-발주시스템)를 통해 제출해야 합니다. (입찰서는 전자적으로만 제출)  
- 제출요건: 입찰서와 제안서를 모두 제출해야 유효합니다.  
- 파일 형식/용량: 제안서 일체는 PDF 파일 형식으로 제출해야 하며 총 용량은 200MB를 초과

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다
[근거: 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp, 한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp, 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp]

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다
[근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)]

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
이 요청은 제가 판정해드릴 수 없습니다.

대신 제공된 문서에 근거한 확인 항목(체크리스트)을 간단히 정리합니다. 귀사 상황을 아래 항목들과 대조해 모두 충족하는지 확인해 주세요.

필수 확인항목(문서 근거)
1. 부정당업자 해당 여부: 지방자치단체를 당사자로 하는 계약에 관한 법령(제92조)에 따른 제한에 해당하지 않아야 함.  
2. 주된 영업소 소재지: 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시여야 함(지방자치단체 법령 근거).  
3. 나라장터(G2B) 등록: 입찰서 제출마감일 전까지 나라장터에 "소프트웨어사업자(업종코드 1468)"로 입찰참가자격 등록되어 있어야 함.  
4. 기업규모 제한: 소프트웨어산업 진흥법 등 관련 지침에 따라 대기업·중견기업(및 상호출자제한기업집단 소속회사)은 참여 불가(중소기업에 해당해야 함).  
5. 직접생산확인증명서: 정보시스템개발서비스(세부품명번호 8111159901)에 대한 ‘직접생산확인증명서’를 입찰마감 전일까지 발급받아 유효기간 내에 있어야 함.  
6. 공동수급·하도급 제한: 공동수급(공동이행방식) 불허, 하도급 불허 — 단독으로 수행 가능해야 함.  
7. 제안서·서류 관련 준수: 제출서류(청렴서약 등)를 성실히 작성·제출해야 하고 허위기재 시 불이익 발생 가능.

각 항목에 대해 필요한 증빙서류(사업자등록증, 본사 소재 증빙, 나라장터 등록화면 캡처, 직접생산확인증명서 등)를 준비해 대조하면 입찰적격 여부를 판단할 수 있습니다. 보다 구체적 판단이 필요하면 위 항목에 대한 귀사의 실제 자료를 제공해 주세요(단, 제가 최종적 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
답변할 수 없다.

해당 문서들에서는 제안서 평가의 세부내용·결과를 공개하지 않거나(비공개 원칙), 제안서의 추가 제안·발표내용이 동일한 효력을 가진다고 명시되어 있어(따라서 평가위원 개별의 비공개 중점사항을 문서 밖에서 단정할 근거가 없음) 문서 범위 내에서 '평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트'를 특정·추정하여 제공할 수 없습니다. 필요하다면 평가 우선순위를 파악하는 데 필요한 객관적 근거(예: 평가기준 및 배점표 원문, 평가위원 지침·회의록 등)를 확인해 달라고 요청해 주세요.  
[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp]

[dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
문서만으로 수주확률(퍼센트)을 계산해 달라는 요청에는 답변할 수 없습니다.

판단에 필요한 주요 정보(간단히 참고용 목록):
- 귀사의 관련 수행실적(유사 사업 경험, 레퍼런스)
- 제출 예정 제안서(기술·가격) 내용
- 평가 기준 및 배점 상세(기술/가격/실무진 등)
- 입찰참가자(경쟁사) 현황 및 예상 제안 수준
- 재무 상태·조직 인력 가용성(파견요건 충족 여부)
- 입찰 보증금·제출서류 충족 여부

위 항목들이 제공되면 문서 근거와 대조해 판단에 필요한 조건 목록을 정리해 드릴 수 있습니다.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다.

[근거: 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp]

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
실시간 정보는 제공된 문서에서 확인할 수 없습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [47]:
def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]

    if numbers:
        all_numbers_match = True
        for num in numbers:
            if num in answer_norm:
                continue
            if len(num) == 4 and num.startswith('20'):
                if num[2:] in answer_norm:
                    continue
            num_no_zero = re.sub(r'^0+', '', num)
            if num_no_zero and num_no_zero in answer_norm:
                continue
            all_numbers_match = False
            break

        if not all_numbers_match:
            return False

        raw_words = re.split(r'[\s,·:()]+', fact_text)
        stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
        core_words = []
        for w in raw_words:
            w = w.rstrip('.,')
            if len(w) < 2:
                continue
            if re.match(r'^\d', w):
                continue
            for suf in stopwords_suffix:
                if w.endswith(suf) and len(w) > len(suf):
                    w = w[:-len(suf)]
                    break
            w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
            if len(w) >= 2:
                core_words.append(w)

        if not core_words:
            return True

        match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
        return match_count / max(len(core_words), 1) >= 0.2

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는',
                    '해드릴 수 없', '드릴 수 없']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [48]:
for r in final_c19_check_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in final_c19_check_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 100.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 100.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 100.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[d

In [49]:
final_c19_check_56 = []
for item in rag56:
    question = item['question']
    answer = ask_rfp_v9(question, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_c19_check_56.append({'case_id': item['case_id'], 'answer': answer})
    print(f"[{item['case_id']}] {question}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c01] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함)  
[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c02] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024. 10. 31. 까지 완료해야 합니다.
[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[supplemental-qa-c03] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000천원 (부가세 포함) — 즉 1,515,000,000원입니다.  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c04] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰  
낙찰 절차(선정 방식): 협상에 의한 계약(협상 통해 사업자 선정) — 평가방식은 기술평가 90% / 가격평가 10%로 산정

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c05] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월입니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c06] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
181,913,000원 — VAT 포함입니다.  
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c07] 인천공항운영서비스가 회계·인사 등 경영업무를 통합할 차세대 시스템을 구축하는 데 걸리는 기간은 얼마인가요?
계약 체결일로부터 9개월(안정화 기간 포함)

[근거: 인천공항운영서비스

In [50]:
for r in final_c19_check_56:
    item = next(it for it in rag56 if it['case_id'] == r['case_id'])
    matched, total = check_required_facts(r['answer'], item['gold'].get('required_fact_groups'))
    r['score'] = round(matched / total * 100, 2) if total else None
    print(f"[{r['case_id']}] 점수: {r['score']}")

valid_scores_56 = [r['score'] for r in final_c19_check_56 if r['score'] is not None]
print(f"\nrag-56 전체 평균: {sum(valid_scores_56)/len(valid_scores_56):.2f}/100 ({len(valid_scores_56)}개)")

[supplemental-qa-c01] 점수: 100.0
[supplemental-qa-c02] 점수: 100.0
[supplemental-qa-c03] 점수: 100.0
[supplemental-qa-c04] 점수: 100.0
[supplemental-qa-c05] 점수: 100.0
[supplemental-qa-c06] 점수: 100.0
[supplemental-qa-c07] 점수: 100.0
[supplemental-qa-c08] 점수: 100.0
[supplemental-qa-c09] 점수: 100.0
[supplemental-qa-c10] 점수: 100.0
[supplemental-qa-c11] 점수: 100.0
[supplemental-qa-c12] 점수: 66.67
[supplemental-qa-c13] 점수: 100.0
[supplemental-qa-c14] 점수: 100.0
[supplemental-qa-c15] 점수: 66.67
[supplemental-qa-c16] 점수: 50.0
[supplemental-qa-c18] 점수: 100.0
[supplemental-qa-c19] 점수: 0.0
[supplemental-qa-c20] 점수: 0.0
[supplemental-qa-c23] 점수: 100.0
[supplemental-qa-c25] 점수: 50.0
[supplemental-qa-g01] 점수: 100.0
[supplemental-qa-g02] 점수: 100.0
[supplemental-qa-g03] 점수: 50.0
[supplemental-qa-g04] 점수: 100.0
[supplemental-qa-g05] 점수: 100.0
[supplemental-qa-g06] 점수: 100.0
[supplemental-qa-g07] 점수: 100.0
[supplemental-qa-g08] 점수: 80.0
[supplemental-qa-g11] 점수: 33.33
[supplemental-qa-g12] 점수: 75.0
[supplemental-qa-

In [51]:
item_c19 = next(it for it in rag56 if it['case_id'] == 'supplemental-qa-c19')
answer_c19 = next(r['answer'] for r in final_c19_check_56 if r['case_id'] == 'supplemental-qa-c19')

print("required_fact_groups:")
for group in item_c19['gold'].get('required_fact_groups'):
    print(f"  {group}")
print()

matched, total = check_required_facts(answer_c19, item_c19['gold'].get('required_fact_groups'))
print(f"매칭 결과: {matched}/{total}")

required_fact_groups:
  ['평택시 BIS와 인천공항 ERP는 기술 80% 가격 20%']
  ['한국철도공사 국방과학연구소 GKL 한국농어촌공사는 기술 90% 가격 10%']
  ['비율 차이의 이유가 명시되어 있지 않음']

매칭 결과: 0/3


In [52]:
import inspect
print(inspect.getsource(check_required_facts))

def check_required_facts(answer: str, required_facts) -> tuple[int, int]:
    """required_facts: [["한영대학교"], ["1억", "100,000,000원"], ...] 형태
    (사실 하나당 인정 가능한 표현들의 리스트). 답변 문자열에 각 사실 그룹 중
    하나라도 들어있으면 그 사실을 "맞춘 것"으로 센다.
    반환값: (맞춘 사실 수, 전체 사실 수).

    [2026-09-02 수정] "VAT 포함", "부가가치세 별도" 처럼 두 단어로 된 variant를
    지금까지는 통째로 하나의 연속 문자열로 찾았는데, golden-set-v3-share
    supplemental-qa-c06/c08 재실행 결과를 보니 이게 오탐(사실은 맞는데 Fail
    처리)을 낸다: gpt-5-mini는 "부가가치세(VAT) 포함" 처럼 괄호로 뜻을
    풀어 쓰거나 "부가가치세는 별도" 처럼 조사(는/은/이/가)를 자연스럽게
    끼워 넣는데, 그러면 "부가가치세 포함"/"부가가치세 별도"가 연속 문자열로
    안 걸린다. 심지어 golden set 자신의 reference_answer("VAT가 포함된")도
    이 기준으로는 스스로 통과 못 하는 걸 확인했다 - 즉 연속 문자열 요구 자체가
    채점 기준의 결함이었다. 그래서 variant를 공백 기준으로 단어 단위로 쪼갠
    뒤, 그 단어들이 (순서/인접 여부와 무관하게) 답변 어딘가에 각각 들어있으면
    인정하도록 완화했다. 기존에 연속 문자열로 통과하던 케이스는 이 조건도
    당연히 만족하므로 패스->fail로 역행하는 경우는 없다.

    [2026-09-02 추가 수정] 위 완화를 적용한 뒤 재실행한 supplemental-qa-c02가
    여전히 Fail이었다 - required_fact_groups는 날짜를 ["2024년 10월 31일",
    "2024-10-31", "202

In [53]:
final_c19_check_set13 = []
set_items_full = golden_v3[golden_v3['source_lane'] == 'set']

for _, row in set_items_full.iterrows():
    q = row['query']
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_c19_check_set13.append({'id': row['id'], 'query': q, 'answer': answer, 'expected': row['expected_doc_id']})
    print(f"[{row['id']}] {q}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b1] 철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요?
사업명: 철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역
발주기관: 국가철도공단

[근거: 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp)]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b10] 코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요?
예약발매시스템 개량 ISMP 용역 (예약발매시스템 개량을 위한 업무 종합계획(ISMP) 수립)

[근거: 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp]

[supplemental-set-b12] 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘
다음은 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 목록입니다.

- 한영대학교: 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 (사업예산: 130,000,000원, 기간: 계약일로부터 3개월)  
- 고려대학교: 차세대 포털·학사 정보시스템 구축 사업 (사업예산: 11,270,000,000원, 기간: 계약일로부터 24개월)  
- 서울시립대학교: 학업성취도 다차원 종단분석 통합시스템 1차 고도화 (사업예산: 242,900,000원, 기간: 계약일로부터 5개월)  
- 경희대학교: 산학협력단 정보시스템(인포21) 운영 용역업체 선정 (사업예산: 400,000,000원, 기간: 계약체결일~2026.04.30)  
- 전북대학교: JST 공유대학(원) xAPI 기반 LRS 시스템 구축 (사업예산: 116,000,000원, 기간: 계약체결일로부터 75일)  
- 광주과학기술원: 학사시스템 기능개선 사업 (사업예산: 157,300,000원, 기간: 계약체결일로부터 5개월)  
- 광주과학기술원: 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업 (사업예산: 54,450,000원, 기간: 계약체결일로부터 3개월)  
- 을지대학교: 비교과시스템 개발 (사업예산: 확인되지 않습니다)  
- 대전대학교: 다층적 융합 학습경험 플랫폼(MILE) 구축 (사업예산: 60,000,000원, 기간: 계약일로부터 2개월)  
- 서영대학교 산학협력단: 차세대 교육혁신지원시스템 3단계 구축 사업 (사업예산: 950,000,000원, 기간: 계약일로부

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b3] 병원 전산 장애나 재해에 대비해 진료정보를 복구할 체계를 마련하는 적십자 관련 용역은 무엇인가요?
적십자병원 병원정보 재해복구시스템 구축 용역

[근거: 대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-set-b4] 학생들의 교과 밖 활동을 관리할 온라인 환경을 새로 만드는 을지대 사업은 무엇인가요?
을지대학교 비교과시스템 개발 사업

[근거: 을지대학교_을지대학교 비교과시스템 개발.hwp]



In [54]:
def evaluate_set_answer(expected_docs, answer_text):
    all_docs_in_answer = [doc for doc, _ in all_filenames_with_biz if doc in answer_text]
    expected_set = set(expected_docs)
    found_set = set(all_docs_in_answer)
    tp = len(expected_set & found_set)
    fp = len(found_set - expected_set)
    fn = len(expected_set - found_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1, tp, fp, fn

results_c19_set13 = []
for item in final_c19_check_set13:
    p, r, f1, tp, fp, fn = evaluate_set_answer(item['expected'], item['answer'])
    results_c19_set13.append({'id': item['id'], 'precision': p, 'recall': r, 'f1': f1})
    print(f"[{item['id']}] P={p:.2f} R={r:.2f} F1={f1:.2f}")

avg_p = sum(r['precision'] for r in results_c19_set13) / len(results_c19_set13)
avg_r = sum(r['recall'] for r in results_c19_set13) / len(results_c19_set13)
avg_f1 = sum(r['f1'] for r in results_c19_set13) / len(results_c19_set13)
print(f"\n평균 Precision: {avg_p:.3f}, 평균 Recall: {avg_r:.3f}, 평균 F1: {avg_f1:.3f}")

[supplemental-set-b1] P=1.00 R=1.00 F1=1.00
[supplemental-set-b10] P=1.00 R=1.00 F1=1.00
[supplemental-set-b12] P=1.00 R=1.00 F1=1.00
[supplemental-set-b14] P=1.00 R=1.00 F1=1.00
[supplemental-set-b15] P=0.80 R=1.00 F1=0.89
[supplemental-set-b16] P=1.00 R=1.00 F1=1.00
[supplemental-set-b20] P=1.00 R=1.00 F1=1.00
[supplemental-set-b21] P=1.00 R=1.00 F1=1.00
[supplemental-set-b22] P=1.00 R=1.00 F1=1.00
[supplemental-set-b23] P=1.00 R=1.00 F1=1.00
[supplemental-set-b24] P=1.00 R=1.00 F1=1.00
[supplemental-set-b3] P=1.00 R=1.00 F1=1.00
[supplemental-set-b4] P=1.00 R=1.00 F1=1.00

평균 Precision: 0.985, 평균 Recall: 1.000, 평균 F1: 0.991


In [55]:
for it in core40 + rag56:
    q = it['question']
    if re.search(r'다음\s*\d+\s*(개|건)', q):
        print(f"[{it['case_id']}] {q}")

[supplemental-qa-c19] 다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다.


In [56]:
# "비교" 관련 질문 중 실제로 언급된 사업명이 3개 초과인 문항이 있는지 확인
for it in core40 + rag56:
    q = it['question']
    if '비교' in q:
        # 대략적인 사업명 개수 추정: 쉼표로 나열된 항목 수
        comma_items = q.count(',')
        if comma_items >= 3:
            print(f"[{it['case_id']}] (쉼표 {comma_items}개) {q}")

[supplemental-qa-c19] (쉼표 6개) 다음 6개 사업의 기술평가와 가격평가 비율을 비교해주세요. 평가비율이 같은 사업끼리 묶고, 비율 차이의 이유가 각 문서에 나와 있는지도 알려주세요. 비교 대상은 평택시 BIS 구축사업, 인천공항운영서비스 차세대 ERP 구축사업, 한국철도공사 운행정보기록 자동분석시스템 개량 사업, 국방과학연구소 기록관리시스템 통합 활용 및 보안 환경 구축 사업, GKL 그룹웨어 구축사업, 한국농어촌공사 네팔 수자원관리 Pilot 시스템 구축용역입니다.


In [57]:
for _, row in set_items_full.iterrows():
    q = row['query']
    if '비교' in q or ('공통점' in q and '차이점' in q):
        comma_items = q.count(',')
        print(f"[{row['id']}] (쉼표 {comma_items}개) {q}")

[supplemental-set-b23] (쉼표 0개) 봉화군 재난통합관리시스템과 충북연구원 재난안전데이터 사업의 공통점과 차이점은?
